# End-to-End Demo

This notebook walks through the local pipeline from API health check to reranked retrieval and grounded answer generation.

It uses `configs/serving_api_demo_mock.yaml`, which points at a tiny mock Ollama-compatible server on port `11435`, so the notebook can run even if the real Ollama app is not installed.


In [ ]:
from pathlib import Path
import sys
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
from pprint import pprint

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from fastapi.testclient import TestClient
from serving.api import create_app

CONFIG_PATH = REPO_ROOT / "configs" / "serving_api_demo_mock.yaml"
CONFIG_PATH


In [ ]:
class MockOllamaHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        return

    def _send(self, payload, status=200):
        import json

        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/api/tags":
            self._send({"models": [{"name": "phi3:mini"}]})
            return
        self._send({"error": "not found"}, status=404)

    def do_POST(self):
        import json

        if self.path != "/api/chat":
            self._send({"error": "not found"}, status=404)
            return

        length = int(self.headers.get("Content-Length", "0"))
        payload = json.loads(self.rfile.read(length).decode("utf-8"))
        prompt = payload["messages"][-1]["content"].lower()

        if "what causes inflation" in prompt:
            answer = (
                "Inflation can happen when aggregate demand rises faster than supply [1]. "
                "Cost-push inflation also occurs when higher wages or input prices lead firms to raise prices [2]."
            )
        elif "what is a corporation" in prompt:
            answer = "A corporation is a legal entity that is separate from its owners [1]."
        else:
            answer = "I don't know."

        self._send(
            {
                "model": payload.get("model", "phi3:mini"),
                "message": {"role": "assistant", "content": answer},
                "done": True,
                "done_reason": "stop"
            }
        )


mock_server = HTTPServer(("127.0.0.1", 11435), MockOllamaHandler)
mock_thread = threading.Thread(target=mock_server.serve_forever, daemon=True)
mock_thread.start()
print("Mock Ollama server listening on http://127.0.0.1:11435")


In [ ]:
app = create_app(str(CONFIG_PATH))
client_ctx = TestClient(app)
client = client_ctx.__enter__()
print("FastAPI TestClient is ready")


## Health Check

The `/health` endpoint confirms that the FAISS index, retrieval models, and generator wiring are all ready.


In [ ]:
health = client.get("/health")
health.raise_for_status()
pprint(health.json())


## Retrieval Only

This calls `/retrieve` so you can inspect the reranked passages without generation.


In [ ]:
retrieve_response = client.post(
    "/retrieve",
    json={"query": "what causes inflation", "top_k": 2},
)
retrieve_response.raise_for_status()
retrieve_payload = retrieve_response.json()
pprint(retrieve_payload)


## End-to-End Query

This calls `/query`, which runs FAISS retrieval, cross-encoder reranking, and generator-backed answer synthesis.


In [ ]:
query_response = client.post(
    "/query",
    json={
        "query": "what is a corporation",
        "top_k": 2,
        "include_prompt": True,
    },
)
query_response.raise_for_status()
query_payload = query_response.json()
print(query_payload["answer"])
print("Citations:", query_payload["citation_ids"])
print()
print(query_payload["prompt"])


## Reference Metrics

For quick context, here is the current comparison table generated earlier in the project.


In [ ]:
results_table_path = REPO_ROOT / "results" / "results_table.md"
print(results_table_path.read_text(encoding="utf-8"))


## Cleanup

Run this at the end of the notebook to release the in-memory client and mock server.


In [ ]:
client_ctx.__exit__(None, None, None)
mock_server.shutdown()
mock_server.server_close()
print("Clean shutdown complete")
